Pivot table + Heatmap

 Step 1: Pivot format: Load data from SQL server to python

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

In [4]:
# Connection (apna server/database naam daalo)
engine = create_engine(
    "mssql://DESKTOP-GVEVCJJ/Olist_Raw_Data"
        "?driver=ODBC+DRIVER+17+FOR+SQL+SERVER"
     "&trusted_connection=yes"
)


In [6]:
# Long format data SQL se load karo
cohort_query = """
WITH customer_first_order AS (
    SELECT
        dc.customer_unique_id,
        MIN(o.order_purchase_ts) AS first_order_date
    FROM clean.orders o
    JOIN clean.customers c ON c.customer_id = o.customer_id
    JOIN dim_customer dc ON dc.customer_unique_id = c.customer_unique_id
    GROUP BY dc.customer_unique_id
),
cohort_base AS (
    SELECT customer_unique_id,
        DATEFROMPARTS(YEAR(first_order_date), MONTH(first_order_date), 1) AS cohort_month
    FROM customer_first_order
),
customer_orders AS (
    SELECT
        dc.customer_unique_id,
        DATEFROMPARTS(YEAR(o.order_purchase_ts), MONTH(o.order_purchase_ts), 1) AS order_month
    FROM clean.orders o
    JOIN clean.customers c ON c.customer_id = o.customer_id
    JOIN dim_customer dc ON dc.customer_unique_id = c.customer_unique_id
),
cohort_activity AS (
    SELECT
        cb.cohort_month,
        DATEDIFF(MONTH, cb.cohort_month, co.order_month) AS month_offset,
        COUNT(DISTINCT co.customer_unique_id) AS active_customers
    FROM cohort_base cb
    JOIN customer_orders co ON co.customer_unique_id = cb.customer_unique_id
    GROUP BY cb.cohort_month, DATEDIFF(MONTH, cb.cohort_month, co.order_month)
),
cohort_size AS (
    SELECT cohort_month, active_customers AS cohort_total
    FROM cohort_activity WHERE month_offset = 0
)
SELECT
    ca.cohort_month,
    ca.month_offset,
    ca.active_customers,
    cs.cohort_total,
    CAST(ca.active_customers * 100.0 / cs.cohort_total AS DECIMAL(5,2)) AS retention_pct
FROM cohort_activity ca
JOIN cohort_size cs ON cs.cohort_month = ca.cohort_month
WHERE ca.cohort_month >= '2017-01-01'   -- 2016 sparse cohorts exclude
  AND ca.cohort_month <= '2018-08-01'   -- incomplete end months exclude
ORDER BY ca.cohort_month, ca.month_offset
"""
cohort_df = pd.read_sql(cohort_query, engine)
print(f"Rows loaded: {len(cohort_df)}")
print(f"Cohorts: {cohort_df['cohort_month'].nunique()}")
print(f"Max month_offset: {cohort_df['month_offset'].max()}")

Rows loaded: 211
Cohorts: 20
Max month_offset: 19


Step 2: Heatmap

In [ ]:
# Pivot table banana — rows=cohort_month, columns=month_offset, values=retention_pct
cohort_pivot = cohort_df.pivot_table(
    index='cohort_df',
    columns='month_offset',
    values='retention_pct'
)

# cohort_month ko readable format mein convert karo
cohort_pivot.index = pd.to_datetime(cohort_pivot.index).strftime('%b %Y')


